In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd", "Catalog")
dbutils.widgets.text("schema", "elina_sharabura", "Schema")
dbutils.widgets.text("num_files", "1000", "Number of files to generate")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
NUM_FILES = int(dbutils.widgets.get("num_files"))

SOURCE_FILE = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/events.csv"
TARGET_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/streaming_source/events"
TIMESTAMP_COLUMN = "timestamp"

dbutils.fs.mkdirs(TARGET_DIR)

#### Inspect source schema

In [0]:
df_preview = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(SOURCE_FILE)
)


#### Split into simulated stream files

In [0]:
import pandas as pd
import numpy as np

df = pd.read_csv(SOURCE_FILE)
df = df.sort_values(TIMESTAMP_COLUMN)
chunks = np.array_split(df, NUM_FILES)

#### Release function

In [0]:
def release_batch(start, end):
    for i in range(start, end):
        chunks[i].to_csv(f"{TARGET_DIR}/events_part_{i:04d}.csv", index=False)
    return f"Released files {start}–{end - 1}"

In [0]:
release_batch(0, 300)

In [0]:
release_batch(300, 600)

#### Schema evolution

In [0]:
chunks[600] = chunks[600].copy()
chunks[600]["device_type"] = "mobile"

release_batch(600, 700)

##### Adding a new column

##### Observing the rescued-data column

In [0]:
chunks[700] = chunks[700].copy()
chunks[700].iloc[0, chunks[700].columns.get_loc("qty")] = "invalid_value"

release_batch(700, 800)

##### Renaming a column

In [0]:
chunks[800] = chunks[800].copy()
chunks[800] = chunks[800].rename(columns={"event_type": "event_category"})

release_batch(800, 900)

##### Final wave release remaining files

In [0]:
release_batch(900, 1000)